##### Importing Data from KAGGLE

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iamsouravbanerjee/customer-shopping-trends-dataset",output_dir='/Users/rakshandha/Desktop/code/python projects/customer_behaviour/data')

/Users/rakshandha/envs/tutorial/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dataset with Pandas

In [1]:
import pandas as pd

df = pd.read_csv('/Users/rakshandha/Desktop/code/python projects/customer_behaviour/data/shopping_trends_updated.csv')

Inspecting and Understanding the Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe(include='all')

In [ ]:
df.isna().sum()

Standardizing Column Names

In [2]:
df.columns = (
    df.columns.str.lower()
    .str.replace(' ','_')
)

In [3]:
df.rename(columns={'purchase_amount_(usd)':'purchase_amount'},inplace=True)

In [ ]:
df.columns.to_list()

Creating Age Groups

In [4]:
labels = ['Young Adult','Adult','Middle-Aged','Senior']

df['age_groups'] = pd.qcut(
    df['age'],
    q=4,
    labels=labels
    )

In [ ]:
df[['age','age_groups']].sample(10)

Creating purchase_frequency Column for Better Analysis

In [5]:
df['frequency_of_purchases'].drop_duplicates()

0        Fortnightly
2             Weekly
4           Annually
6          Quarterly
10         Bi-Weekly
15           Monthly
20    Every 3 Months
Name: frequency_of_purchases, dtype: str

In [6]:
frequency_mapping = {
    'Fortnightly':14,
    'Weekly':7,
    'Annually':365,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Monthly':30,
    'Every 3 Months':90
}

# mapping
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [7]:
df[ ['frequency_of_purchases','purchase_frequency_days'] ].drop_duplicates()

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
2,Weekly,7
4,Annually,365
6,Quarterly,90
10,Bi-Weekly,14
15,Monthly,30
20,Every 3 Months,90


Promo Code and Discount Applied Column Inspection

In [8]:
df[['discount_applied','promo_code_used']].drop_duplicates()

,discount_applied,promo_code_used
0,Yes,Yes
1677,No,No


In [9]:
(df['discount_applied']== df['promo_code_used']).all()

np.True_

In [10]:
# as both columns show same values 
df = df.drop(columns='promo_code_used')

In [11]:
pip install psycopg2-binary sqlalchemy


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
from sqlalchemy import create_engine, text

username = "rakshandha"
password = "Raaqshas.2828"
host = "localhost"
port = "5432"
database = "customer_analysis"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Test connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print("Connected to:", result.fetchone()[0])

# Load dataframe
table_name = "customer"

df.to_sql(
    table_name,
    engine,
    if_exists="replace",
    index=False
)

print(f"Data successfully loaded into table '{table_name}'.")

Connected to: customer_analysis
Data successfully loaded into table 'customer'.
